In [44]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import numpy as np
from torch import nn
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from tqdm import tqdm

In [45]:
df = pd.read_parquet("../../data/processed/features.parquet")
df.head()

,features,label
0,"[[0.257080078125, 0.2939453125, 0.320068359375...",1
1,"[[0.305908203125, 0.34912109375, 0.3740234375,...",2
2,"[[0.2548828125, 0.301025390625, 0.3330078125, ...",3
3,"[[0.240966796875, 0.2900390625, 0.333984375, 0...",4
4,"[[0.1929931640625, 0.238037109375, 0.241943359...",5


In [46]:
df.iloc[0]['features']

array([array([0.25708008, 0.29394531, 0.32006836, 0.33007812, 0.3359375 ,
              0.24902344, 0.28100586, 0.30395508, 0.3190918 , 0.22900391,
              0.26098633, 0.28710938, 0.30493164, 0.2199707 , 0.24902344,
              0.27392578, 0.29199219, 0.2199707 , 0.24304199, 0.26391602,
              0.28100586, 0.67089844, 0.65283203, 0.62695312, 0.60693359,
              0.59179688, 0.60009766, 0.57617188, 0.56787109, 0.56591797,
              0.60009766, 0.57910156, 0.57080078, 0.56884766, 0.60302734,
              0.58300781, 0.57519531, 0.57177734, 0.60693359, 0.58789062,
              0.58007812, 0.57714844, 0.        , 0.        , 0.        ,
              0.        , 0.        , 0.        , 0.        , 0.        ,
              0.        , 0.        , 0.        , 0.        , 0.        ,
              0.        , 0.        , 0.        , 0.        , 0.        ,
              0.        , 0.        , 0.        , 0.        , 0.        ,
              0.        , 0.        , 

In [47]:
max_seq_len = 0
for row in df['features']:
    if len(row) > max_seq_len:
        max_seq_len = len(row)
print(f"Max sequence lenght: {max_seq_len}")

Max sequence lenght: 100


In [48]:
max_seq_len = 100
for row in df['features']:
    tensors = [torch.from_numpy(a).float() for a in row]
    row_tensor = torch.stack(tensors)
    print(row_tensor.shape, end=" ")
    if len(row_tensor) < max_seq_len:
        pad_size = max_seq_len - len(row_tensor)
        padded = torch.cat([
            row_tensor, 
            torch.zeros(pad_size, 84)
        ], dim=0)
    elif len(row_tensor) > max_seq_len:
        padded = row_tensor[:max_seq_len]
    else:
        padded = row_tensor
    print(padded.shape)
    

torch.Size([40, 84]) torch.Size([100, 84])
torch.Size([57, 84]) torch.Size([100, 84])
torch.Size([62, 84]) torch.Size([100, 84])
torch.Size([49, 84]) torch.Size([100, 84])
torch.Size([74, 84]) torch.Size([100, 84])
torch.Size([51, 84]) torch.Size([100, 84])
torch.Size([68, 84]) torch.Size([100, 84])
torch.Size([74, 84]) torch.Size([100, 84])
torch.Size([37, 84]) torch.Size([100, 84])
torch.Size([75, 84]) torch.Size([100, 84])
torch.Size([51, 84]) torch.Size([100, 84])
torch.Size([83, 84]) torch.Size([100, 84])
torch.Size([70, 84]) torch.Size([100, 84])
torch.Size([65, 84]) torch.Size([100, 84])
torch.Size([66, 84]) torch.Size([100, 84])
torch.Size([66, 84]) torch.Size([100, 84])
torch.Size([59, 84]) torch.Size([100, 84])
torch.Size([62, 84]) torch.Size([100, 84])
torch.Size([61, 84]) torch.Size([100, 84])
torch.Size([69, 84]) torch.Size([100, 84])
torch.Size([51, 84]) torch.Size([100, 84])
torch.Size([61, 84]) torch.Size([100, 84])
torch.Size([39, 84]) torch.Size([100, 84])
torch.Size(

C:\Users\Марсель\AppData\Local\Temp\ipykernel_22548\2214042556.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:212.)
  tensors = [torch.from_numpy(a).float() for a in row]


torch.Size([57, 84]) torch.Size([100, 84])
torch.Size([57, 84]) torch.Size([100, 84])
torch.Size([57, 84]) torch.Size([100, 84])
torch.Size([57, 84]) torch.Size([100, 84])
torch.Size([57, 84]) torch.Size([100, 84])
torch.Size([57, 84]) torch.Size([100, 84])
torch.Size([37, 84]) torch.Size([100, 84])
torch.Size([37, 84]) torch.Size([100, 84])
torch.Size([37, 84]) torch.Size([100, 84])
torch.Size([37, 84]) torch.Size([100, 84])
torch.Size([37, 84]) torch.Size([100, 84])
torch.Size([37, 84]) torch.Size([100, 84])
torch.Size([37, 84]) torch.Size([100, 84])
torch.Size([37, 84]) torch.Size([100, 84])
torch.Size([37, 84]) torch.Size([100, 84])
torch.Size([37, 84]) torch.Size([100, 84])
torch.Size([58, 84]) torch.Size([100, 84])
torch.Size([58, 84]) torch.Size([100, 84])
torch.Size([58, 84]) torch.Size([100, 84])
torch.Size([58, 84]) torch.Size([100, 84])
torch.Size([58, 84]) torch.Size([100, 84])
torch.Size([58, 84]) torch.Size([100, 84])
torch.Size([58, 84]) torch.Size([100, 84])
torch.Size(

In [49]:
class LandmarkDataset(Dataset):
    def __init__(self, dataset_path: str, max_seq_len: int = None):
        self.df = pd.read_parquet(dataset_path)
        if max_seq_len is None:
            max_seq_len = 0
            for row in self.df['features']:
                if len(row) > max_seq_len:
                    max_seq_len = len(row)
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        features = self.df.iloc[idx]['features']
        label = self.df.iloc[idx]['label']
        
        tensors = [torch.from_numpy(a).float() for a in features]
        row_tensor = torch.stack(tensors)
        
        if len(row_tensor) < max_seq_len:
            pad_size = max_seq_len - len(row_tensor)
            padded = torch.cat([
                row_tensor, 
                torch.zeros(pad_size, 84)
            ], dim=0)
        elif len(row_tensor) > max_seq_len:
            padded = row_tensor[:max_seq_len]
        else:
            padded = row_tensor
        
        return padded, label    

In [50]:
dataset = LandmarkDataset("../../data/processed/features.parquet")

train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size
    
train_dataset, val_dataset = random_split(
        dataset, [train_size, val_size]
)

train_loader = DataLoader(
        train_dataset,
        batch_size=16,
        shuffle=True,
)
    
val_loader = DataLoader(
        val_dataset,
        batch_size=16
)

In [51]:
def train(
    model,
    epochs,
    optimizer,
    loss_fn
):
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for features, labels in (pbar := tqdm(train_loader, desc=f"Epoch {epoch+1:3d}/{epochs} │ Training", leave=False)):
            optimizer.zero_grad()
            logits = model(features)
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * features.size(0)
            total_train += labels.size(0)
            
            preds = torch.argmax(logits, dim=1)
            correct_train += (preds == labels).sum().item()
            
            running_loss = train_loss / total_train
            pbar.set_postfix({"loss": f"{running_loss:.4f}"})

        model.eval()
        val_loss = 0.0
        total_val = 0
        correct_val = 0
        
        with torch.inference_mode():
            for features, labels in (pbar := tqdm(val_loader, desc=f"Epoch {epoch+1:3d}/{epochs} │ Validating", leave=False)):
                logits = model(features)
                loss = loss_fn(logits, labels)
                
                val_loss += loss.item() * features.size(0)
                total_val += labels.size(0)
                
                preds = torch.argmax(logits, dim=1)
                correct_val += (preds == labels).sum().item()
                
                running_val_loss = val_loss / total_val
                pbar.set_postfix({"loss": f"{running_val_loss:.4f}"})
        
        avg_train_loss = train_loss / total_train
        avg_val_loss = val_loss / total_val
        
        train_acc = correct_train / total_train
        val_acc = correct_val / total_val
        
        print(
            f"Epoch {epoch+1:3d}/{epochs} │ "
            f"Train Loss: {avg_train_loss:.4f} │ Val Loss: {avg_val_loss:.4f} │ "
            f"Train Acc: {train_acc:.4f} │ Val Acc: {val_acc:.4f}"
        )

In [52]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

In [53]:
class GestureTransformer(nn.Module):
    def __init__(
        self,
        num_classes: int = 33,
        d_model: int = 84,
        d_ff: int = 256,
        num_encoders: int = 3,
        nheads: int = 4, 
        dropout: float = 0.3 
    ):
        super().__init__()
        
        # Input projection
        self.input_proj = nn.Linear(84, d_model)
        self.pos_encoding = PositionalEncoding(d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Transformer encoder only
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nheads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="relu",
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_encoders
        )
        
        # Classifier
        self.fc = nn.Linear(d_model, num_classes)
    
    def forward(self, x):
        batch_size = x.size(0)
        
        # Input projection
        x = self.input_proj(x)  # (batch, seq_len, d_model)
        x = self.pos_encoding(x)
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # (batch, seq_len+1, d_model)
        
        # Transformer encoder
        encoded = self.transformer(x)  # (batch, seq_len+1, d_model)
        
        # Use CLS token for classification
        cls_output = encoded[:, 0, :]  # (batch, d_model)
        
        # Classification
        logits = self.fc(cls_output)  # (batch, num_classes)
        return logits

In [54]:
class GestureRNN(nn.Module):
    def __init__(
        self,
        num_classes: int = 33,
        d_model: int = 84,
        d_hidden: int = 128,
        num_layers: int = 3,
        dropout: float = 0.3
    ):
        super().__init__()  
        
        self.input_proj = nn.Linear(84, d_model)
        
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=d_hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Linear(d_hidden, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Input: (batch_size, seq_len, 84)
        x = self.input_proj(x)  # (batch_size, seq_len, d_model)
        
        # GRU forward
        output, hidden = self.gru(x)  # output: (batch_size, seq_len, d_hidden)
        
        # Use the last output of the sequence
        last_output = output[:, -1, :]  # (batch_size, d_hidden) - last timestep
        
        # Apply dropout and classify
        last_output = self.dropout(last_output)
        logits = self.fc(last_output)  # (batch_size, num_classes)
        
        return logits

In [55]:
class GestureCNN(nn.Module):
    def __init__(
        self,
        num_classes: int = 33,
        d_model: int = 84,
        d_hidden: int = 256,
        dropout: float = 0.3
    ):
        super().__init__()

        self.input_proj = nn.Linear(84, d_model)  
        
        # Convolution layers
        self.convs = nn.Sequential(
            # After input_proj: (batch, seq_len, d_model)  transpose  (batch, d_model, seq_len)
            nn.Conv1d(d_model, d_hidden, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),  # seq_len: 100 → 50
            
            nn.Conv1d(d_hidden, d_hidden * 2, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),  # seq_len: 50 → 25
            
            nn.Conv1d(d_hidden * 2, d_hidden * 4, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # (batch, d_hidden*4, 1)
        )   
        
        # Classifier - input size must match last conv layer output channels
        self.fc = nn.Linear(d_hidden * 4, num_classes)   
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x shape: (batch_size, seq_len=100, features=84)
        x = self.input_proj(x)  # (batch_size, 100, d_model)
        x = x.transpose(1, 2)   # (batch_size, d_model, 100)
        
        x = self.convs(x)       # (batch_size, d_hidden*4, 1)
        x = x.squeeze(-1)       # (batch_size, d_hidden*4)
        
        x = self.dropout(x)
        x = self.fc(x)          # (batch_size, num_classes)
        return x

In [58]:
print("Transformer")
model = GestureTransformer(
    num_classes=33,
    d_model=32,
    d_ff=64,
    num_encoders=2,
    nheads=2,
    dropout=0.3
)

optimizer = Adam(model.parameters(), lr=1e-3)
loss_fn = CrossEntropyLoss()

example_input = torch.randn(16, 12, 84)
logits = model(example_input)

print(logits.shape)
train(
    model=model,
    epochs=10,
    optimizer=optimizer,
    loss_fn=loss_fn
)

Transformer
torch.Size([16, 33])


Epoch   1/10 │ Train Loss: 3.5104 │ Val Loss: 3.4687 │ Train Acc: 0.0309 │ Val Acc: 0.0405


Epoch   2/10 │ Train Loss: 3.4714 │ Val Loss: 3.4423 │ Train Acc: 0.0380 │ Val Acc: 0.0554


Epoch   3/10 │ Train Loss: 3.4372 │ Val Loss: 3.5145 │ Train Acc: 0.0538 │ Val Acc: 0.0384


Epoch   4/10 │ Train Loss: 3.4221 │ Val Loss: 3.4175 │ Train Acc: 0.0547 │ Val Acc: 0.0518


...

Epoch   6/10 │ Train Loss: 3.3617 │ Val Loss: 3.2861 │ Train Acc: 0.0685 │ Val Acc: 0.0888


Epoch   7/10 │ Train Loss: 3.2812 │ Val Loss: 3.1227 │ Train Acc: 0.0879 │ Val Acc: 0.0945


Epoch   8/10 │ Train Loss: 3.0995 │ Val Loss: 2.9049 │ Train Acc: 0.0980 │ Val Acc: 0.1158


Epoch   9/10 │ Train Loss: 2.9421 │ Val Loss: 2.8169 │ Train Acc: 0.1103 │ Val Acc: 0.1229


Epoch  10/10 │ Train Loss: 2.8653 │ Val Loss: 2.8791 │ Train Acc: 0.1232 │ Val Acc: 0.1314


In [59]:
print("RNN")
model = GestureRNN(
    num_classes=33,
    d_model=84,
    d_hidden=64,
    num_layers=2,
    dropout=0.3
)
optimizer = Adam(model.parameters(), lr=1e-3)
loss_fn = CrossEntropyLoss()

example_input = torch.randn(16, 100, 84)
logits = model(example_input)

print(logits.shape)
train(  
    model=model,
    epochs=10,
    optimizer=optimizer,
    loss_fn=loss_fn
)

RNN
torch.Size([16, 33])


Epoch   1/10 │ Train Loss: 3.4840 │ Val Loss: 3.4706 │ Train Acc: 0.0291 │ Val Acc: 0.0249


Epoch   2/10 │ Train Loss: 3.4709 │ Val Loss: 3.4633 │ Train Acc: 0.0327 │ Val Acc: 0.0391


Epoch   3/10 │ Train Loss: 3.4539 │ Val Loss: 3.4565 │ Train Acc: 0.0421 │ Val Acc: 0.0412


Epoch   4/10 │ Train Loss: 3.4332 │ Val Loss: 3.4077 │ Train Acc: 0.0524 │ Val Acc: 0.0668


...

Epoch   6/10 │ Train Loss: 3.3801 │ Val Loss: 3.3798 │ Train Acc: 0.0620 │ Val Acc: 0.0526


Epoch   7/10 │ Train Loss: 3.3603 │ Val Loss: 3.3857 │ Train Acc: 0.0653 │ Val Acc: 0.0419


Epoch   8/10 │ Train Loss: 3.3509 │ Val Loss: 3.3400 │ Train Acc: 0.0710 │ Val Acc: 0.0682


Epoch   9/10 │ Train Loss: 3.3333 │ Val Loss: 3.3165 │ Train Acc: 0.0735 │ Val Acc: 0.0710


Epoch  10/10 │ Train Loss: 3.3136 │ Val Loss: 3.3670 │ Train Acc: 0.0785 │ Val Acc: 0.0575


In [60]:
print("CNN")
model = GestureCNN(
    num_classes=33,
    d_model=84,
    d_hidden=64,
    dropout=0.3
)
optimizer = Adam(model.parameters(), lr=1e-3)
loss_fn = CrossEntropyLoss()

example_input = torch.randn(16, 100, 84)
logits = model(example_input)

print(logits.shape)
train(
    model=model,
    epochs=10,
    optimizer=optimizer,
    loss_fn=loss_fn
)

CNN
torch.Size([16, 33])


Epoch   1/10 │ Train Loss: 3.4788 │ Val Loss: 3.4637 │ Train Acc: 0.0353 │ Val Acc: 0.0391


Epoch   2/10 │ Train Loss: 3.4478 │ Val Loss: 3.4328 │ Train Acc: 0.0444 │ Val Acc: 0.0405


Epoch   3/10 │ Train Loss: 3.3944 │ Val Loss: 3.3339 │ Train Acc: 0.0531 │ Val Acc: 0.0547


Epoch   4/10 │ Train Loss: 3.2636 │ Val Loss: 3.3094 │ Train Acc: 0.0641 │ Val Acc: 0.0661


...

Epoch   6/10 │ Train Loss: 2.8956 │ Val Loss: 2.9479 │ Train Acc: 0.1195 │ Val Acc: 0.1193


Epoch   7/10 │ Train Loss: 2.7494 │ Val Loss: 2.6367 │ Train Acc: 0.1463 │ Val Acc: 0.1577


Epoch   8/10 │ Train Loss: 2.6733 │ Val Loss: 2.4759 │ Train Acc: 0.1665 │ Val Acc: 0.2067


Epoch   9/10 │ Train Loss: 2.4940 │ Val Loss: 2.3272 │ Train Acc: 0.2031 │ Val Acc: 0.2500


Epoch  10/10 │ Train Loss: 2.3563 │ Val Loss: 2.0964 │ Train Acc: 0.2425 │ Val Acc: 0.3033
